## Appendix A: Matrices

Runs the oloffson estimator on the region 2 and coastwide accuracy assessments and reports the confusion matrices in latex tables.

NOTE: This notebook assumes you have already run the coastwide and region 2 notebooks to get their respective extent maps.

In [7]:
import sys
from pathlib import Path
from string import Template

sys.path.insert(0, "../../..")

from mangrove_mapper.validation.validator import evaluate_model, olofsson_estimate

ROOT = Path("../../..")
AA   = ROOT / "mangrove_mapper/accuracy_assessment"
DATA = ROOT / "data"
REFS = {"Collect Earth Online": "ceo_label", "Planet": "planet_label"}

CASES = {
    "region2": {
        "sheet": AA / "region_2_accuracy_assessment_final.csv",
        "extent": DATA / "2021-extent-xgb-gmw-T50.tif",
        "caption": "Pomeroon River estuary confusion matrices for XGB-GMW model extent "
                "maps with references from CEO and Planet imagery"
    },
    "coastwide": {
        "sheet": AA / "full_coast_accuracy_assessment_final.csv",
        "extent": DATA / "2021-COAST-EXTENT-XGB-GMW-full-T50.tif",
        "caption":"Coast-wide confusion matrices for XGB-GMW model extent maps with "
                "references from CEO and Planet imagery"
    }
}


Latex structure was adapted from https://www.latex4technics.com/?note=3HX2

In [ ]:
TABULAR = Template(r"""    \begin{tabular}{l|l|c|c|c}
    \multicolumn{2}{c}{}&\multicolumn{2}{c}{Reference ($ref)}&\\
    \cline{3-4}
    \multicolumn{2}{c|}{}&Mangrove&Non-Mangrove&\multicolumn{1}{c}{Total}\\
    \cline{2-4}
    \multirow{2}{*}{Map}&Mangrove& $a & $b & $ab\\
    \cline{2-4}
    & Non-Mangrove & $c & $d & $cd\\
    \cline{2-4}
    \multicolumn{1}{c}{} & \multicolumn{1}{c}{Total} & \multicolumn{1}{c}{$ac} & \multicolumn{1}{c}{$bd} & \multicolumn{1}{c}{$N}\\
    \end{tabular}""")

def block(est, ref):
    n = est["confusion"]
    # left Map  right Reference
    a = n.loc["mangrove", "mangrove"]          # correctly mapped mangrove numbwea
    b = n.loc["mangrove", "non_mangrove"]      # commission numbers
    c = n.loc["non_mangrove", "mangrove"]      # omission numbers
    d = n.loc["non_mangrove", "non_mangrove"]
    return TABULAR.substitute(ref=ref, a=a, b=b, c=c, d=d,
                              ab=a + b, cd=c + d, ac=a + c, bd=b + d, N=a + b + c + d)

In [9]:
for name, cfg in CASES.items():
    col = cfg["extent"].stem # model name
    df, meta = evaluate_model(str(cfg["sheet"]), model_name=col,
                              model_extent_map_path=str(cfg["extent"]))
    blocks = []
    for ref, ref_col in REFS.items():
        est = olofsson_estimate(df, map_col=col, weights=meta["weights"],
                                domain_ha=meta["domain_ha"], ref_col=ref_col)
        blocks.append(block(est, ref))
        pc = est["per_class"].loc["mangrove"]
        print(f'{name:10} {ref:20} n={est["n"]}  OA={est["overall_accuracy"]:.3f}  '
              f'UA={pc["users_acc"]:.3f}  PA={pc["producers_acc"]:.3f}')

    tex = "\n".join([
        r"\begin{table}[htbp]",
        rf'    \caption{{{cfg["caption"]}}}',
        r"    \centering", "",
        blocks[0], "",
        r"    \vspace{1.5em}", "",
        blocks[1],
        r"\end{table}"])
    Path(f"confusion_matrices_appendix_{name}.tex").write_text(tex + "\n")
    print(f"confusion_matrices_appendix_{name}.tex\n")

region2    Collect Earth Online n=400  OA=0.965  UA=0.919  PA=0.933
region2    Planet               n=400  OA=0.944  UA=0.912  PA=0.862
confusion_matrices_appendix_region2.tex

coastwide  Collect Earth Online n=600  OA=0.972  UA=0.440  PA=0.859
coastwide  Planet               n=600  OA=0.963  UA=0.390  PA=0.643
confusion_matrices_appendix_coastwide.tex

